### 1. 데이터 통합

In [1]:
import pandas as pd
import json
import re
import os

In [2]:
with open('../data/extracted_jobs_result_wanted.json', 'r', encoding='utf-8') as f: wanted = json.load(f)
with open('../data/extracted_jobs_result_jobkorea.json', 'r', encoding='utf-8') as f: jobkorea = json.load(f)
with open('../data/final_extracted_jobs_result_saramin.json', 'r', encoding='utf-8') as f: saramin = json.load(f)

df_wanted = pd.DataFrame(wanted)
df_jobkorea = pd.DataFrame(jobkorea)
df_saramin = pd.DataFrame(saramin)

print(f"원티드 데이터 개수: {len(df_wanted)}개 / 컬럼: {list(df_wanted.columns)}")
print(f"잡코리아 데이터 개수: {len(df_jobkorea)}개 / 컬럼: {list(df_jobkorea.columns)}")
print(f"사람인 데이터 개수: {len(df_saramin)}개 / 컬럼: {list(df_saramin.columns)}")

원티드 데이터 개수: 4404개 / 컬럼: ['job_id', 'tag_id', 'year_filter', 'company', 'position', 'main_tasks', 'requirements', 'preferred', 'skill_tags', 'location', 'scraped_date', 'hard_skills', 'soft_skills', 'preferences', 'culture_keywords', 'urgency_score', 'urgency_reason']
잡코리아 데이터 개수: 10107개 / 컬럼: ['공고번호', '회사명', '공고제목', '연차/경력', '지역', '기술스택/분야', '마감일', '상세내용', 'hard_skills', 'soft_skills', 'preferences', 'culture_keywords', 'urgency_score', 'urgency_reason']
사람인 데이터 개수: 25950개 / 컬럼: ['공고번호', '원문', 'hard_skills', 'soft_skills', 'preferences', 'culture_keywords', 'urgency_score', 'urgency_reason']


In [3]:
# urgency_score 샘플 형태 확인
print("[원티드] urgency_score 샘플:")
if 'urgency_score' in df_wanted.columns:
    print(df_wanted['urgency_score'].unique()[:10])
else:
    print("'urgency_score' 컬럼이 원티드에 없습니다.")

print("\n[잡코리아] urgency_score 샘플:")
if 'urgency_score' in df_jobkorea.columns:
    print(df_jobkorea['urgency_score'].unique()[:10])
else:
    print("'urgency_score' 컬럼이 잡코리아에 없습니다.")

print("\n[사람인] urgency_score 샘플:")
if 'urgency_score' in df_saramin.columns:
    print(df_saramin['urgency_score'].unique()[:10])
else:
    print("'urgency_score' 컬럼이 사람인에 없습니다.")

[원티드] urgency_score 샘플:
[4 3 2]

[잡코리아] urgency_score 샘플:
[3 2 1 4 5]

[사람인] urgency_score 샘플:
[3 1 4 2 5]


In [4]:
print()
if 'urgency_score' in df_wanted.columns:
    print(df_wanted['urgency_score'].value_counts().sort_index())


urgency_score
2       4
3    2269
4    2131
Name: count, dtype: int64


In [5]:
# 특수문자, 공백, 이메일 등 제거
def clean_for_model(text):
    if not text or pd.isna(text):
        return ""
    # URL, 이메일 주소 제거
    text = re.sub(r'http\S+|www\S+|<[^>]*>', ' ', str(text))
    text = re.sub(r'\S+@\S+', ' ', text)
    # 한글, 영어, 숫자, 기본 문장부호만 유지
    text = re.sub(r'[^가-힣a-zA-Z0-9\s.,!?~]', ' ', text)
    # 연속된 공백 하나로 통합
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [6]:
# 원티드 전처리(position + main_tasks + requirements + preferred)
def process_wanted(df):
    rows = []
    df_valid_data = df[df['urgency_score'].notna()].copy()
    
    for _, row in df_valid_data.iterrows():
        full_text = f"{row.get('position', '')} {row.get('main_tasks', '')} {row.get('requirements', '')} {row.get('preferred', '')}"
        
        rows.append({
            'text': clean_for_model(full_text),
            'label': int(float(row['urgency_score'])),
            'platform': 'wanted'
        })
    return pd.DataFrame(rows)

In [7]:
# 잡코리아 전처리(공고제목 + 상세내용)
def process_jobkorea(df):
    rows = []
    df_valid_data = df[df['urgency_score'].notna()].copy()
    
    for _, row in df_valid_data.iterrows():
        full_text = f"{row.get('공고제목', '')} {row.get('상세내용', '')}"
        
        rows.append({
            'text': clean_for_model(full_text),
            'label': int(float(row['urgency_score'])),
            'platform': 'jobKorea'
        })
    return pd.DataFrame(rows)

In [8]:
# 사람인 전처리(원문)
def process_jobkorea(df):
    rows = []
    df_valid_data = df[df['urgency_score'].notna()].copy()
    
    for _, row in df_valid_data.iterrows():
        full_text = f"{row.get('원문', '')}"
        
        rows.append({
            'text': clean_for_model(full_text),
            'label': int(float(row['urgency_score'])),
            'platform': 'saramin'
        })
    return pd.DataFrame(rows)

In [9]:
# 전처리 실행
df_wanted_processed = process_wanted(df_wanted)
df_jobkorea_processed = process_jobkorea(df_jobkorea)
df_saramin_processed = process_jobkorea(df_saramin)

print(f'원티드 전처리 완료: {len(df_wanted_processed)}건')
print(f'잡코리아 전처리 완료: {len(df_jobkorea_processed)}건')
print(f'사람인 전처리 완료: {len(df_saramin_processed)}건')

원티드 전처리 완료: 4404건
잡코리아 전처리 완료: 10107건
사람인 전처리 완료: 25950건


In [10]:
# 데이터셋 하나로 합치기
df_combine_all = pd.concat([df_wanted_processed, df_jobkorea_processed, df_saramin_processed], ignore_index=True)

# 중복 데이터 제거
df_combine_all = df_combine_all.drop_duplicates(subset=['text']).reset_index(drop=True)

print('\n통합 데이터셋 내 플랫폼별 지분:')
print(df_combine_all['platform'].value_counts())

print('\n통합 데이터셋 내 라벨별 지분:')
print(df_combine_all['label'].value_counts().sort_index())


통합 데이터셋 내 플랫폼별 지분:
platform
saramin    25178
wanted      3698
Name: count, dtype: int64

통합 데이터셋 내 라벨별 지분:
label
1      277
2      449
3    24463
4     3391
5      296
Name: count, dtype: int64


### 2. 시급성 점수 보정

In [ ]:
# 점수 강제 고정할 키워드
HARD_RULES = {
    5: ['즉시', '급구', '긴급 채용', '즉시 출근', '당일 면접', '빠른 입사', '시급함', 
        '긴급모집', '즉시지원', '즉시 지원', '즉시 입사', '즉시입사', '긴급 수립', '당장'],
    1: ['상시 채용', '인재 풀', '인재풀', '상시 모집', '인력풀 등록', 
        '인재 pool', '인재 등록', '채용 시까지',
        '서류 보관', '상시 오픈', '순차 진행', '순차 검토', '순차적 검토', 
        '적격자 채용시 마감', '적격자 선발 시', '적격자 채용 시', 
        ]
}

# 점수 미세 조정할 키워드
SOFT_RULES = {
    +1: ['충원', '추가 모집', '확장 채용', '신속한 진행', '대규모 채용', 
        '지속 충원', '빠른 전형', '사업 확장', '인원 확장', '충원 예정', 
        '조기 마감', '우선 검토', '신속 검토', '빠른 합격', '적극 채용'],
    -1: ['상시', '순차적', '수시', '순차', 
        '서류 접수 순', '개별 연락', 
        
        '체험형 인턴', '전환형 인턴', 
        '수습 기간', '수습 3개월', '어소시에이트', '교육생',
        
        '풀(pool)', 'pool 등록', '상시운영', '상시 확보'
        ]
}

In [39]:
def adjust_urgency_by_keywords(text, current_label):

    if not isinstance(text, str) or text.strip() == "":
        return current_label
    
    # 공백 정제 및 소문자화
    clean_txt = text.lower()
    
    # 강제 고정 규칙 적용 1) -> 5점
    for keyword in HARD_RULES[5]:
        if keyword in clean_txt:
            return 5
    
    # 강제 고정 규칙 적용 2) -> 1점 
    for keyword in HARD_RULES[1]:
        if keyword in clean_txt:
            return 1
            
    # 미세 조정 규칙 적용
    adjusted_score = current_label
    for boost, keywords in SOFT_RULES.items():
        for keyword in keywords:
            if keyword in clean_txt:
                adjusted_score += boost
                break # 동일 카테고리 내 중복 차단
                
    # 점수 범위를 1 ~ 5점 사이로 제한 (안전장치)
    return max(1, min(5, adjusted_score))

In [40]:
df_adjusted = df_combine_all.copy()

# 기존 점수 백업(비교용)
df_adjusted['original_label'] = df_adjusted['label']

# 3. 보정 함수 실행 적용
df_adjusted['label'] = df_adjusted.apply(
    lambda row: adjust_urgency_by_keywords(row['text'], row['label']), 
    axis=1
)

changed_mask = df_adjusted['original_label'] != df_adjusted['label']
df_changed = df_adjusted[changed_mask]

print(f"🔄 키워드 규칙 기반으로 라벨이 자동 보정된 공고 수: {len(df_changed)}개 (전체 중 {len(df_changed)/len(df_adjusted)*100:.1f}%)")

# 5. 어떻게 보정되었는지 샘플 5개 확인
if len(df_changed) > 0:
    print("\n🔎 [실제 라벨 보정 샘플 데이터 확인]")
    display(df_changed[['text', 'original_label', 'label']].head(50))
else:
    print("\n⚠️ 보정된 데이터가 없습니다. 키워드가 본문에 들어있는지 확인해보세요.")

🔄 키워드 규칙 기반으로 라벨이 자동 보정된 공고 수: 11432개 (전체 중 39.6%)

🔎 [실제 라벨 보정 샘플 데이터 확인]


,text,original_label,label
56,Santa Frontend Engineer 3년 이상 이런 업무를 담당합니다. 산타...,3,2
65,글로벌 해운선사IT솔루션 ITO업무 및 차세대 솔루션 도입 API개발 계약직 글로벌...,4,5
75,커넥트 웹 React 개발자 커넥트의 빠른 성장에 따른 확장성 있고 생산성 있는 웹...,3,2
94,AEM개발자 Adobe AEM 개발 및 프로젝트 제안 리딩 Adobe AEM 연동을...,3,5
95,해운선사용 IT솔루션 Allegro 개발 및 운영 글로벌 해운 선사용 IT 솔루션 ...,3,5
102,글로벌 해운선사IT솔루션 ITO업무 및 차세대 솔루션 도입 API개발 글로벌 해운 ...,4,5
137,"엔터프라이즈플랫폼유닛 Front end 리테일플랫폼 올리브영의 온라인, 오프라인, ...",3,2
138,커머스플랫폼유닛 Front end 커머스 플랫폼 안정적이고 일관된 구조 위에서 서비...,4,3
156,Back end Software Engineer 4년 이상 이러한 경험을 하실 수 ...,3,2
157,Front end Software Engineer 4년 이상 이러한 경험을 하실 수...,3,2


In [42]:
print("="*60)
print("📊 [보정 전] 1~5점 시급성 점수 분포:")
print(df_adjusted['original_label'].value_counts().sort_index())
print("-"*60)
print("🔥 [보정 후] 1~5점 시급성 점수 분포:")
print(df_adjusted['label'].value_counts().sort_index())
print("="*60)

📊 [보정 전] 1~5점 시급성 점수 분포:
original_label
1      277
2      449
3    24463
4     3391
5      296
Name: count, dtype: int64
------------------------------------------------------------
🔥 [보정 후] 1~5점 시급성 점수 분포:
label
1      720
2     9350
3    14810
4     2896
5     1100
Name: count, dtype: int64


### 3. 통합 데이터 저장

In [43]:
OUTPUT_FILE = './data/integrated_learning_dataset.csv'

df_combine_all.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f'{OUTPUT_FILE}로 통합 데이터 저장 완료')

df_combine_all.head()

./data/integrated_learning_dataset.csv로 통합 데이터 저장 완료


,text,label,platform
0,개발 데이터 마이그레이션 매니저 동물병원 전자차트와 반려동물 헬스케어 플랫폼의 미래...,4,wanted
1,인턴 프롭테크 플랫폼 서비스 개발 어떤 일을 하나요? 기존 서비스를 Next.js ...,3,wanted
2,ASP.NET 개발자 신입 주요업무 웹 어플리케이션 개발 및 유지보수 개발 환경 프...,3,wanted
3,"프론트엔드 개발자 신입 4년, Next.js React Typescript 유리프트...",3,wanted
4,인턴 프론트엔드 개발자 프론티아 웹 모바일 서비스의 신규 기능 개발 및 개선 디자이...,3,wanted
